# Model Selection

## Overview

Model selection identifies which predictors to include and which model family best fits the data. It balances goodness-of-fit against complexity to avoid overfitting.

**Approaches:**

| Approach | Basis | Use when |
|---|---|---|
| AIC/BIC | Penalised likelihood | Comparing nested or non-nested models |
| Likelihood ratio test | Nested model comparison | Formal hypothesis test |
| Cross-validation | Out-of-sample prediction | Predictive accuracy |
| Stepwise selection | Sequential addition/removal | Quick exploration (use cautiously) |

**AIC vs BIC:**
- AIC: asymptotically selects the best predictive model (may include noise variables)
- BIC: penalises complexity more heavily; consistent for true model under certain conditions
- In practice: use AIC for prediction, BIC for parsimony

**Important:** all selection methods should be followed by diagnostic checks on the selected model.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
from itertools import combinations
from scipy import stats

rng = np.random.default_rng(42)
n = 250
elevation   = rng.uniform(50, 400, n)
nitrate     = rng.gamma(2, 2, n)
phosphorus  = 0.6*nitrate + rng.normal(0, 0.5, n)  # correlated with nitrate
temperature = 15 - 0.02*elevation + rng.normal(0, 1, n)
treatment   = rng.choice([0,1], n)
noise1      = rng.normal(0, 1, n)  # irrelevant
noise2      = rng.normal(0, 1, n)  # irrelevant
richness    = (25 - 0.04*elevation - 0.8*nitrate + 2.5*treatment
               + rng.normal(0, 3, n))
df = pd.DataFrame(dict(richness=richness, elevation=elevation,
    nitrate=nitrate, phosphorus=phosphorus, temperature=temperature,
    treatment=treatment, noise1=noise1, noise2=noise2))
print(f"n={n}, true predictors: elevation, nitrate, treatment")

---
## AIC and BIC Comparison

In [ ]:
candidates = {
    'null':    'richness ~ 1',
    'elev':    'richness ~ elevation',
    'nitr':    'richness ~ nitrate',
    'treat':   'richness ~ C(treatment)',
    'elev+nitr': 'richness ~ elevation + nitrate',
    'true':    'richness ~ elevation + nitrate + C(treatment)',
    'full':    'richness ~ elevation + nitrate + phosphorus + temperature + C(treatment)',
    'kitchen': 'richness ~ elevation + nitrate + phosphorus + temperature + C(treatment) + noise1 + noise2',
}
results = []
for name, formula in candidates.items():
    m = smf.ols(formula, df).fit()
    results.append(dict(model=name, k=m.df_model+2,
                        AIC=m.aic, BIC=m.bic, R2=m.rsquared_adj))
res_df = pd.DataFrame(results).set_index('model').sort_values('AIC')
res_df['dAIC'] = res_df['AIC'] - res_df['AIC'].min()
res_df['dBIC'] = res_df['BIC'] - res_df['BIC'].min()
print(res_df.round(2))
print("\ndAIC < 2: substantial support; dAIC > 10: no support")

---
## Likelihood Ratio Test

In [ ]:
# LRT: compare nested models formally
m_reduced = smf.ols('richness ~ elevation + nitrate', df).fit()
m_full    = smf.ols('richness ~ elevation + nitrate + C(treatment)', df).fit()
# LRT statistic
LRT_stat = -2 * (m_reduced.llf - m_full.llf)
df_diff  = m_full.df_model - m_reduced.df_model
LRT_p    = stats.chi2.sf(LRT_stat, df=df_diff)
print(f"Reduced model (elevation + nitrate):          AIC={m_reduced.aic:.2f}")
print(f"Full model    (+ treatment):                  AIC={m_full.aic:.2f}")
print(f"LRT: chi2={LRT_stat:.3f}, df={df_diff:.0f}, p={LRT_p:.4f}")
print(f"{'Adding treatment significantly improves fit' if LRT_p<0.05 else 'Treatment does not improve fit'}")
print("\nLRT is only valid for nested models (reduced is a special case of full)")

---
## Best Subset Selection

In [ ]:
predictors = ['elevation','nitrate','phosphorus','temperature','treatment','noise1','noise2']
subset_results = []
for k in range(1, len(predictors)+1):
    for combo in combinations(predictors, k):
        formula = 'richness ~ ' + ' + '.join(
            [f'C({v})' if v=='treatment' else v for v in combo])
        m = smf.ols(formula, df).fit()
        subset_results.append(dict(k=k, predictors='+'.join(combo),
                                    AIC=m.aic, BIC=m.bic, R2adj=m.rsquared_adj))
sub_df = pd.DataFrame(subset_results)
best_aic = sub_df.loc[sub_df.groupby('k')['AIC'].idxmin()]
best_bic = sub_df.loc[sub_df.groupby('k')['BIC'].idxmin()]
fig, axes = plt.subplots(1,2,figsize=(12,4))
for ax, df_best, metric, color in [(axes[0],best_aic,'AIC','steelblue'),
                                     (axes[1],best_bic,'BIC','#e74c3c')]:
    ax.plot(df_best['k'], df_best[metric], 'o-', color=color, lw=2)
    ax.axvline(df_best[metric].idxmin()+1, color='grey', lw=1, linestyle='--')
    ax.set_xlabel('Number of predictors'); ax.set_ylabel(metric)
    ax.set_title(f'Best subset by {metric}')
plt.tight_layout(); plt.show()
print("Best AIC model:", best_aic.loc[best_aic['AIC'].idxmin(), 'predictors'])
print("Best BIC model:", best_bic.loc[best_bic['BIC'].idxmin(), 'predictors'])

In [ ]:
# AIC weights: model averaging
top_models = sub_df.nsmallest(5, 'AIC').copy()
top_models['dAIC'] = top_models['AIC'] - top_models['AIC'].min()
top_models['weight'] = np.exp(-0.5*top_models['dAIC'])
top_models['weight'] /= top_models['weight'].sum()
print("Top 5 models by AIC weight:")
print(top_models[['k','predictors','AIC','dAIC','weight']].round(3).to_string())
print("\nAIC weight = relative evidence for each model")
print("Weight > 0.9 for one model -> strong evidence for that model")
print("Weights spread across many models -> model uncertainty is high")
print("  -> consider model averaging or report uncertainty in selection")

---

## Common Pitfalls

**1. Treating model selection as a hypothesis test**  
AIC and BIC are not p-values. A model with dAIC = 2 is not "significantly better" — it has slightly more empirical support. Similarly, a non-significant LRT does not mean the predictor has no effect; it means the data do not provide strong evidence to distinguish the models.

**2. Running stepwise selection and reporting the final model as if it were pre-specified**  
Stepwise selection performs many implicit tests and exploits random variation in the data. The selected model is biased toward including noise variables, and standard errors for selected coefficients are too small. If stepwise is used for exploration, always validate the selected model on held-out data.

**3. Comparing AIC across models fit on different datasets or with different response transformations**  
AIC values are only comparable when models are fit on exactly the same observations with the same response variable (same scale). A log-transformed response and the untransformed response have different likelihoods and incomparable AIC values.

**4. Ignoring multicollinearity during selection**  
Correlated predictors (elevation and temperature, nitrate and phosphorus) will not both be selected by AIC even if both have genuine effects — their information is partially redundant. Examine VIF and the correlation structure before selection; consider ridge regression or PCA-based approaches when predictors are highly correlated.

**5. Not validating the selected model with diagnostics**  
Model selection identifies which predictors to include, not whether the model's assumptions hold. Always run residual diagnostics on the selected model — a well-selected but misspecified model (wrong family, missing interaction) produces biased inference.
---
*python_methods_library - Samantha McGarrigle*